In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import catboost as cb
from processor import PolarsLoader, ExprProcessor, PandasConverter
from IPython.display import Markdown

In [3]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(3, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')
with open('grade_subgrade.pkl', 'rb') as f:
    c_map = pkl.load(f)
df_train['grade_subgrade_no'] = df_train['grade_subgrade'].map(c_map).astype('int')
df_test['grade_subgrade_no'] = df_test['grade_subgrade'].map(c_map).astype('int')
df_train.shape, df_test.shape

((593994, 13), (254569, 12))

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade_no']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
import importlib
from modeler import Experimenter

In [7]:
e = Experimenter(df_train.sample(frac = 0.01, random_state = 123), sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [8]:
Markdown(
    e.desc_spec()
)

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=3, random_state=123, shuffle=True)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=123)` |
| **Splitter Params** | `{y='loan_paid_back'}` |
| **Outer Folds** | 3 |
| **Inner Folds** | 1 |

In [9]:
e.add_grp('clf', role = 'exp', parent_grp = None, edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', role = 'pipe', parent_grp = None, method = 'transform')

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.set_node('ohe', 'preprocessor', OneHotEncoder, edges = [(None, X_cat)], params={'sparse_output': False})
e.build_pipeline()

🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
✅ Build complete!


In [11]:
e.build_pipeline()

🔄 Building 0 node(s)
✅ Build complete!


In [12]:
e.build_pipeline(rebuild=True)

🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
✅ Build complete!


In [13]:
from analyzer import Stacker

In [14]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression)

In [15]:
from modeler import col
e.set_node('lr1', 'lr', edges = [('std', None)])
e.set_node('lr2', 'lr', edges = [('std', None), ('ohe', col.ohe_drop_first)])

In [16]:
e.build_experiment('lr*')

🔄 Building 2 node(s)
  ├─ Building 'lr1'...
  ├─ Building 'lr2'...
  ├─ Building 'lr1'...
  ├─ Building 'lr2'...
  ├─ Building 'lr1'...
  ├─ Building 'lr2'...
✅ Build complete!


In [17]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [18]:
Markdown(
    e.desc_node('lr2', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr2["lr2"]
        lr2_dummy[ ]
        style lr2_dummy fill:none,stroke:none
    end
    style node_lr2 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr2
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr2
    node_std --> node_lr2
```

**Path from Root to 'lr2' (3 path(s) found)**

In [19]:
e.add_grp('dim_reduction', parent_grp = 'preprocessor')

In [20]:
e.nodes['lr1'].get_result('coef').T.groupby(level = -1).mean().T

,intercept,std__annual_income,std__credit_score,std__debt_to_income_ratio,std__grade_subgrade_no,std__interest_rate,std__loan_amount
0,1.659772,-0.024023,0.803259,-0.755152,0.151995,0.029723,-0.026436


In [21]:
e.nodes['lr2'].get_result('coef').T.groupby(level = -1).mean().T

,intercept,ohe__education_level_High School,ohe__education_level_Master's,ohe__education_level_Other,ohe__education_level_PhD,ohe__employment_status_Retired,ohe__employment_status_Self-employed,ohe__employment_status_Student,ohe__employment_status_Unemployed,ohe__gender_Male,...,ohe__loan_purpose_Vacation,ohe__marital_status_Married,ohe__marital_status_Single,ohe__marital_status_Widowed,std__annual_income,std__credit_score,std__debt_to_income_ratio,std__grade_subgrade_no,std__interest_rate,std__loan_amount
0,2.693546,0.045385,-0.056036,0.184038,0.786244,1.711976,-0.097732,-3.177548,-4.863163,0.065691,...,0.568845,-0.216559,-0.227785,0.258193,-0.011718,1.061754,-0.867006,0.277145,-0.00823,-0.018095


In [22]:
from sklearn.decomposition import PCA
e.set_node('pca', 'dim_reduction', processor=PCA, edges = [('std', None)], params={'n_components': 0.9})

In [23]:
e.build_pipeline()

🔄 Building 1 node(s)
  ├─ Building 'pca'...
  ├─ Building 'pca'...
  ├─ Building 'pca'...
✅ Build complete!


In [24]:
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.build_pipeline()

  └─ Effeced 3 dependent node(s): ['lr1', 'lr2', 'pca']
🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'pca'...
  ├─ Building 'std'...
  ├─ Building 'pca'...
  ├─ Building 'std'...
  ├─ Building 'pca'...
✅ Build complete!


In [25]:
e.build_experiment('lr*')

🔄 Building 2 node(s)
  ├─ Building 'lr1'...
  ├─ Building 'lr2'...
  ├─ Building 'lr1'...
  ├─ Building 'lr2'...
  ├─ Building 'lr1'...
  ├─ Building 'lr2'...
✅ Build complete!


In [26]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [27]:
e.set_node('lr3', 'lr', edges = [('ohe', None), ('pca', None)])

In [28]:
Markdown(
    e.desc_node('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [29]:
Markdown(
    e.desc_node('lr3', direction = 'LR', show_params=True)
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr>"]
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>PCA</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>n_components</b></td><td align='left'>0.9</td></tr></table>"]
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [30]:
from analyzer import Stacker
s = Stacker(e, target_edge = (None, [y]), output_var = slice(0, -1))

In [31]:
from modeler._metric import Metric
from sklearn.metrics import roc_auc_score
m = Metric('AUC', e, target_edge = (None, [y]), output_var = slice(0, -1), metric_func = roc_auc_score, include_train = True)

In [32]:
m.get_metric(0, 'lr1')

0  train  train_0    0.218787
          valid_0    0.235036
   valid             0.222494
dtype: float64

In [33]:
from modeler._stacker import Stacker
s = Stacker(e, target_edge = (None, [y]), output_var = slice(0, -1))

In [34]:
s.get_stack(1, 'lr1')

(array([[0.43345928],
        [0.17256913],
        [0.09328277],
        ...,
        [0.0901328 ],
        [0.10364243],
        [0.14536954]]),
 ['lr1__loan_paid_back_0'])

In [35]:
e.add_grp('cb', parent_grp='clf', processor=cb.CatBoostClassifier, params={'verbose': 0})

In [36]:
e.set_node('cb1',  grp = 'cb', edges = [(None, X_num), (None, X_cat)], params = {'cat_features': X_cat})

In [37]:
import lightgbm as lgb

In [38]:
e.add_grp('lgb', parent_grp='clf', processor=lgb.LGBMClassifier, params={'verbose': -1})

In [39]:
e.set_node('lgb1',  grp = 'lgb', edges = [(None, X_num), (None, X_cat)], params={'categorical_features': X_cat})

In [40]:
e.build_experiment(".*")

🔄 Building 3 node(s)
  ├─ Building 'lr3'...
  ├─ Building 'cb1'...
  ├─ Building 'lgb1'...
  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0820, valid_1-binary_logloss: 0.2943  ├─ Building 'lr3'...
  ├─ Building 'cb1'...


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


  ├─ Building 'lgb1'...
  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0752, valid_1-binary_logloss: 0.2759  ├─ Building 'lr3'...
  ├─ Building 'cb1'...


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


  ├─ Building 'lgb1'...
  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0738, valid_1-binary_logloss: 0.2509✅ Build complete!


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


In [41]:
e.desc_node_vars('cb1', 0)

(                          name
 node seq                      
 Root 0           annual_income
      1    debt_to_income_ratio
      2            credit_score
      3             loan_amount
      4           interest_rate
      5       grade_subgrade_no
      6                  gender
      7          marital_status
      8         education_level
      9       employment_status
      10           loan_purpose,
                     name
 0  cb1__loan_paid_back_0
 1  cb1__loan_paid_back_1)

In [42]:
e.nodes['cb1'].objs_[0][0][0].obj.evals_result_['validation_1']

{'Logloss': [0.645759441246397,
  0.6130065322875022,
  0.5725020987063175,
  0.5380210558266334,
  0.5149434214939483,
  0.48901834322816917,
  0.4658994928719027,
  0.44223958047939793,
  0.42341022944119033,
  0.40600890202352247,
  0.3896425477733485,
  0.3774130169578602,
  0.3663380082866546,
  0.3559981144487788,
  0.34591239985819444,
  0.33750671270033883,
  0.3304182506563109,
  0.3235921489014019,
  0.316973691886976,
  0.3126578450679723,
  0.3086949796067106,
  0.30496437044762953,
  0.30027004431766324,
  0.29787817742148065,
  0.2956063172061011,
  0.2936174437478543,
  0.29141830382467176,
  0.28936286889222007,
  0.28646100287579246,
  0.28382234089728553,
  0.2814016968297356,
  0.2787614831277691,
  0.2777716419933787,
  0.27603540913433566,
  0.2748312398649252,
  0.2743936373089017,
  0.2726353836552194,
  0.2711474006610633,
  0.2701325128905084,
  0.2686668357820882,
  0.26772497398803313,
  0.26714451524166843,
  0.2665280284599556,
  0.26592385103015403,
  0.26

In [43]:
Markdown(
    e.desc_node('cb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_cb1["cb1"]
        cb1_dummy[ ]
        style cb1_dummy fill:none,stroke:none
    end
    style node_cb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_cb1
```

**Path from Root to 'cb1' (1 path(s) found)**

In [44]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [45]:
Markdown(
    e.desc_node('lgb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lgb1["lgb1"]
        lgb1_dummy[ ]
        style lgb1_dummy fill:none,stroke:none
    end
    style node_lgb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_lgb1
```

**Path from Root to 'lgb1' (1 path(s) found)**

In [46]:
e._find_descendants('std')

{'lr1', 'lr2', 'lr3', 'pca'}

In [47]:
# e = Experimenter(df_train, sp = skf, sp_v = None, splitter_params = {'y': y})

In [48]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [49]:
from modeler import create_like

In [50]:
e2 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = ss, sp_v = StratifiedKFold(2, random_state=123, shuffle = True),
                splitter_params = {'y': y})

🔄 Creating new Experimenter with same structure...
   ├─ Created base Experimenter with 1 fold(s)
   ├─ Cloned 6 group(s)
   └─ Cloned 8 node(s)
✅ Structure cloning complete!


In [51]:
e2.build_pipeline()
e2.build_experiment('.*')

🔄 Building 3 node(s)
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'pca'...
✅ Build complete!
🔄 Building 5 node(s)
  ├─ Building 'lr1'...
  ├─ Building 'lr2'...
  ├─ Building 'lr3'...
  ├─ Building 'cb1'...
  ├─ Building 'lgb1'...
  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0426, valid_1-binary_logloss: 0.3078✅ Build complete!


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


In [52]:
e2.nodes['lgb1'].get_result("evals_result")

[[{'training': OrderedDict([('binary_logloss',
                 [np.float64(0.44342973313223016),
                  np.float64(0.4047031546974569),
                  np.float64(0.37506265245490317),
                  np.float64(0.3510635956014906),
                  np.float64(0.33085574211078217),
                  np.float64(0.3134371942510402),
                  np.float64(0.2979221296092706),
                  np.float64(0.2843609489781788),
                  np.float64(0.2722169823014045),
                  np.float64(0.2613176890574337),
                  np.float64(0.25139892208646325),
                  np.float64(0.24237036100899517),
                  np.float64(0.23461964467610388),
                  np.float64(0.2272579497531432),
                  np.float64(0.22073066850559833),
                  np.float64(0.2145442131836554),
                  np.float64(0.2091616432838145),
                  np.float64(0.20371716977375437),
                  np.float64(0.19819801062408

In [53]:
for (true_train, true_train_v), (prd_train, prd_train_v) in  zip(
    e.get_node_train_output(0, None, [y]),
    e.get_node_train_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_train.data, prd_train.data), 
        roc_auc_score(true_train_v.data, prd_train_v.data)
    )

0.9411658860340462 0.9119514435171505


In [54]:
for true_valid, prd_valid in  zip(
    e.get_node_valid_output(0, None, [y]),
    e.get_node_valid_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_valid.data, prd_valid.data)
    )

0.9253651719043245


In [55]:
class Metric:
    def __init__(
        self, e, target_edge, output_var, metric_func, include_train = False
    ):
        self.e = e
        self.target_edge = target_edge
        self.output_var = output_var
        self.include_train = include_train
        self.metric_func = metric_func
        self.result = {}
        self.build_ids = {}

    def calc_idx(self, nodes, idx):
        result = {}
        grps = {}
        if self.include_train:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and (node, idx) in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_output(idx, None, [y]), self.e.get_node_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for no, ((true_train, true_valid), (prd_train, prd_valid)) in enumerate(iterator):
                    result_train = self.metric_func(true_train[0].data, prd_train[0].data)
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)
                    if true_train[1] is not None:
                        result_sub = {
                            (idx, 'train', f'train_{no}'): result_train,
                            (idx, 'train', f'valid_{no}'): self.metric_func(true_train[1].data, prd_train[1].data),
                            (idx, 'valid', ''): result_valid
                        }
                    else:
                        result_sub = {
                            (idx, 'train'): result_train, (idx, 'valid'): result_valid
                        }
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)
        else:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and node in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_valid_output(idx, None, [y]), self.e.get_node_valid_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for true_valid, prd_valid in iterator:
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)                    
                    result_sub = {idx: result_valid}
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)

        c, mx = None, -1
        for i in grps.values():
            i = i[::-1]
            if c is None:
                c = i
            else:
                mx = max(mx, len(i))
                for j in range(min(len(c), len(i))):
                    if c[j] != i[j]:
                        c = i[:j]
                        break
            if len(c) == 0:
                break
        for k, i in grps.items():
            i = tuple([''] * (mx - len(i) - len(c)) +  i[:-len(c)] + [k])
            result[k] = result[k].rename(i)
        return pd.DataFrame(result.values())

    def calc(self, nodes):
        result = [self.calc_idx(nodes, i) for i in range(self.e.get_n_splits())]
        return pd.concat(result, axis=1)

In [56]:
for (y_true_train_t, y_true_valid_t), y_true_valid in e2.get_node_output(0, 'cb1'):
    print(y_true_train_t)
    print(y_true_valid_t)

In [57]:
e.nodes['cb1'].objs_[0][0][0].X_

['annual_income',
 'debt_to_income_ratio',
 'credit_score',
 'loan_amount',
 'interest_rate',
 'grade_subgrade_no',
 'gender',
 'marital_status',
 'education_level',
 'employment_status',
 'loan_purpose']

In [58]:
import analyzer
from analyzer import Metric
importlib.reload(analyzer)

<module 'analyzer' from '/home/sun9sun9/jnote/sunkusun9/kaggle/PGS5/PGS5_ep11/analyzer/__init__.py'>

In [59]:
m = Metric(e, (None, [y]), slice(-1, None), roc_auc_score, True)
m.set_nodes('clf')
result = m.get_metric()
result

AttributeError: 'Node' object has no attribute 'grp_name'

In [65]:
e3 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = skf, sp_v = None, splitter_params = {'y': y})

🔄 Creating new Experimenter with same structure...
   ├─ Created base Experimenter with 3 fold(s)
   ├─ Cloned 6 group(s)
[std] Building: 3/3 (100%) ✓ Complete
[lr1] Building: 3/3 (100%) ✓ Complete
[ohe] Building: 3/3 (100%) ✓ Complete
[lr2] Building: 3/3 (100%) ✓ Complete
[pca] Building: 3/3 (100%) ✓ Complete
[lr3] Building: 3/3 (100%) ✓ Complete
[cb1] Building: 3/3 (100%) ✓ Complete
[lgb1] Building: 3/3 (100%) ✓ Complete
   └─ Cloned 8 node(s)
✅ Structure cloning complete!


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


In [73]:
df_input, df_output = e2.desc_node_vars('lr3', 0)
display(df_input)
df_output

name
node seq                                      
ohe  0                      ohe__gender_Female
     1                        ohe__gender_Male
     2                       ohe__gender_Other
     3            ohe__marital_status_Divorced
     4             ohe__marital_status_Married
     5              ohe__marital_status_Single
     6             ohe__marital_status_Widowed
     7         ohe__education_level_Bachelor's
     8        ohe__education_level_High School
     9           ohe__education_level_Master's
     10             ohe__education_level_Other
     11               ohe__education_level_PhD
     12        ohe__employment_status_Employed
     13         ohe__employment_status_Retired
     14   ohe__employment_status_Self-employed
     15         ohe__employment_status_Student
     16      ohe__employment_status_Unemployed
     17             ohe__loan_purpose_Business
     18                  ohe__loan_purpose_Car
     19   ohe__loan_purpose_Debt consolidation
     20            ohe__loan_purpose_Education
     21                 ohe__loan_purpose_Home
     22              ohe__loan_purpose_Medical
     23                ohe__loan_purpose_Other
     24             ohe__loan_purpose_Vacation
pca  0                               pca__pca0
     1                               pca__pca1
     2                               pca__pca2
     3                               pca__pca3
     4                               pca__pca4

,name
0,lr3__loan_paid_back_0
1,lr3__loan_paid_back_1


In [74]:
from analyzer import Stacker
s = Stacker(e3, (None, [y]), slice(-1, None))

In [75]:
s.set_nodes('cb')
s.set_nodes('lr')
s.get_dataset().data

,loan_paid_back,lr2__loan_paid_back_1,cb1__loan_paid_back_1,lr3__loan_paid_back_1,lr1__loan_paid_back_1
id,,,,,
176836,1,0.887501,0.470297,0.714372,0.565392
321322,0,0.882068,0.916742,0.949351,0.852699
225907,0,0.879394,0.486456,0.746855,0.648991
290104,1,0.912323,0.588010,0.743183,0.575475
453658,1,0.915982,0.937856,0.936601,0.824478
...,...,...,...,...,...
425863,1,0.887879,0.986788,0.960933,0.913895
103194,1,0.893064,0.979759,0.963263,0.938460
252332,0,0.067854,0.006536,0.007383,0.459041


In [76]:
e3.nodes['lr3'].objs_[0][0][0].obj.classes_

array([0, 1], dtype=int8)

In [77]:
lr_a.set_nodes('lr')

In [78]:
for inner_idx, df in lr_a.result[('lr1', 0)].items():
    print(type(df['intercept']) == pd.Series, df['intercept'].to_frame().columns)

True Index(['intercept'], dtype='object')


In [81]:
lr_a.get_coef('lr1').T.groupby(level = [2]).mean().T

,std__annual_income,std__credit_score,std__debt_to_income_ratio,std__grade_subgrade_no,std__interest_rate,std__loan_amount
0,-0.015401,0.771408,-0.772686,0.122688,0.034967,-0.046015


In [83]:
lr_a.get_intercept('lr1').T.groupby(level = [2]).mean().T

,intercept
0,1.662578


In [83]:
e.root.data

,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back,grade_subgrade_no
id,,,,,,,,,,,,,
176836,53894.898438,0.201,635,17084.410156,13.72,Male,Married,Bachelor's,Employed,Debt consolidation,D4,1,18
321322,19748.630859,0.051,621,12982.080078,14.51,Female,Married,Master's,Employed,Debt consolidation,D1,0,15
225907,17096.009766,0.190,648,18019.500000,11.64,Female,Single,Bachelor's,Employed,Debt consolidation,D5,0,19
290104,83433.921875,0.222,667,24015.429688,13.32,Female,Married,Bachelor's,Employed,Business,D1,1,15
453658,18492.880859,0.153,706,23961.570312,13.51,Male,Single,Bachelor's,Employed,Car,C3,1,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
425863,105467.007812,0.099,709,4671.779785,11.36,Male,Single,Bachelor's,Employed,Debt consolidation,C3,1,12
103194,14441.030273,0.084,728,18582.089844,11.99,Female,Married,Bachelor's,Employed,Debt consolidation,C5,1,14
252332,14201.660156,0.275,661,29165.949219,13.08,Female,Single,Master's,Unemployed,Debt consolidation,D4,0,18


In [69]:
for i in e2.get_node_valid_output(0, 'cb1', slice(0, -1)):
    print(i.data)

        cb1__loan_paid_back_0
id                           
437626               0.031421
56509                0.989099
590576               0.497218
469144               0.006575
118806               0.131097
...                       ...
167513               0.079113
14614                0.990822
490101               0.158133
341032               0.011666
247234               0.106048

[1188 rows x 1 columns]
        cb1__loan_paid_back_0
id                           
437626               0.024687
56509                0.996380
590576               0.397108
469144               0.010977
118806               0.138448
...                       ...
167513               0.100434
14614                0.995543
490101               0.222619
341032               0.011216
247234               0.123429

[1188 rows x 1 columns]


In [70]:
e2.get_data_valid(0, [('lr1', slice(0, -1))])

<generator object Experimenter.get_data_valid.<locals>.ret_data_func at 0x7fdca2593100>

In [71]:
for i in e2.get_data_valid(0, [('lr1', slice(0, -1))]):
    print(i.data)

        lr1__loan_paid_back_0
id                           
437626               0.126048
56509                0.421086
590576               0.519993
469144               0.062336
118806               0.263459
...                       ...
167513               0.184167
14614                0.635248
490101               0.258190
341032               0.082907
247234               0.196475

[1188 rows x 1 columns]
        lr1__loan_paid_back_0
id                           
437626               0.111893
56509                0.444592
590576               0.641100
469144               0.045669
118806               0.308009
...                       ...
167513               0.172526
14614                0.745087
490101               0.251948
341032               0.070369
247234               0.225258

[1188 rows x 1 columns]


In [72]:
e3 = create_like(e, df_train.sample(frac = 0.1, random_state = 123), splitter_params = {'y': y})

🔄 Creating new Experimenter with same structure...
   ├─ Created base Experimenter with 3 fold(s)
   ├─ Cloned 6 group(s)
[std] Building: 3/3 (100%) ✓ Complete
[lr1] Building: 3/3 (100%) ✓ Complete
[ohe] Building: 3/3 (100%) ✓ Complete
[lr2] Building: 3/3 (100%) ✓ Complete
[pca] Building: 3/3 (100%) ✓ Complete
[lr3] Building: 3/3 (100%) ✓ Complete
[cb1] Building: 3/3 (100%) ✓ Complete
  Progress: 70/100 (70.0%)%)

/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


[lgb1] Building: 3/3 (100%) ✓ Complete
   └─ Cloned 8 node(s)
✅ Structure cloning complete!


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


In [73]:
e3.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

⚠️  Updating existing node 'std'
[std] Building: 3/3 (100%) ✓ Complete
  └─ Found 3 dependent node(s): ['lr1', 'lr3', 'pca']
🔄 Rebuilding nodes: ['std', 'lr1', 'lr3', 'pca']
  ├─ Rebuilding 'std'...
[std] Building: 3/3 (100%) ✓ Complete
  ├─ Rebuilding 'lr1'...
[lr1] Building: 3/3 (100%) ✓ Complete
  ├─ Rebuilding 'lr3'...
[lr3] Building: 3/3 (100%) ✓ Complete
  ├─ Rebuilding 'pca'...
[pca] Building: 3/3 (100%) ✓ Complete
✅ Rebuild complete!


In [74]:
e3.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')

In [75]:
e3.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [76]:
e3.set_node('lr1', 'lr')

⚠️  Updating existing node 'lr1'
[lr1] Building: 3/3 (100%) ✓ Complete


In [77]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    #sgpp.PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [78]:
X_all = df_test.columns
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [79]:
e = Experimenter(df_train, sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [80]:
e.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', method = 'transform')

In [81]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

[std] Building: 3/3 (100%) ✓ Complete


In [82]:
for a, b in e.get_data(0, [(None, X_num)]):
    print(a[0].data, b)

shape: (356_396, 5)
┌───────────────┬──────────────────────┬──────────────┬──────────────┬───────────────┐
│ annual_income ┆ debt_to_income_ratio ┆ credit_score ┆ loan_amount  ┆ interest_rate │
│ ---           ┆ ---                  ┆ ---          ┆ ---          ┆ ---           │
│ f32           ┆ f32                  ┆ i16          ┆ f32          ┆ f32           │
╞═══════════════╪══════════════════════╪══════════════╪══════════════╪═══════════════╡
│ 39059.851562  ┆ 0.091                ┆ 683          ┆ 16166.389648 ┆ 8.97          │
│ 14056.129883  ┆ 0.179                ┆ 581          ┆ 15221.790039 ┆ 13.8          │
│ 50874.160156  ┆ 0.078                ┆ 654          ┆ 13830.820312 ┆ 12.82         │
│ 33253.230469  ┆ 0.073                ┆ 595          ┆ 12862.19043  ┆ 13.84         │
│ 30508.470703  ┆ 0.19                 ┆ 718          ┆ 20981.589844 ┆ 12.74         │
│ …             ┆ …                    ┆ …            ┆ …            ┆ …             │
│ 56280.550781  ┆ 0.203

In [83]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [84]:
e.set_node('lr1', 'lr')

[lr1] Building: 3/3 (100%) ✓ Complete


In [85]:
for i in e.get_data_valid(0, [(None, y)]):
    print(i.data)

shape: (197_998,)
Series: 'loan_paid_back' [i8]
[
	0
	1
	1
	1
	0
	…
	1
	1
	1
	1
	1
]


In [86]:
ss = StratifiedKFold(n_splits=3)
for a, b in ss.split(df_train[X_all],  df_train[y]):
    pass

In [87]:
lr = LogisticRegression()
lr.fit(df_train[X_num], df_train[[y]])

/home/sun9sun9/python312/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/sun9sun9/python312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [76]:
df_train.to_pandas()[[y]].shape

(593994, 1)

In [77]:
e.nodes['lr1'].y

'loan_paid_back'